#Junyang You

# Take home exercise 2

In [0]:
df_pit = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True, inferSchema=True)
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv",header=True,inferSchema=True)

In [0]:
display(df_pit)

In [0]:
display(df_drivers)

In [0]:
df_pit.printSchema()
df_drivers.printSchema()

# Q1: What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

###For this question, I used the pit_stops data because it shows pit stop records for drivers in different races. I focused on the milliseconds column since it gives a direct numeric value for pit stop time. First, I grouped the data by raceId and driverId. So I could calculate the average pit stop time for each driver in each race. Then I looked at the race level only and found the minimum and maximum pit stop times to get the fastest and slowest stop in that race. I changed the values from milliseconds to seconds at the end so the output would be easier to understand.

In [0]:
from pyspark.sql.functions import avg, min, max, round

driver_avg_pit = (
    df_pit
    .groupBy("raceId", "driverId")
    .agg(
        round(avg("milliseconds") / 1000, 3).alias("avg_pit_stop_seconds")
    )
)

race_fast_slow = (
    df_pit
    .groupBy("raceId")
    .agg(
        round(min("milliseconds") / 1000, 3).alias("fastest_pit_stop_seconds"),
        round(max("milliseconds") / 1000, 3).alias("slowest_pit_stop_seconds")
    )
)

q1=(
    driver_avg_pit
    .join(
        race_fast_slow,
        on="raceId",
        how="left"
    )
    .join(
        df_drivers.select("driverId", "forename", "surname"),
        on="driverId",
        how="left"
    )
    .select(
        "raceId",
        "driverId",
        "forename",
        "surname",
        "avg_pit_stop_seconds",
        "fastest_pit_stop_seconds",
        "slowest_pit_stop_seconds"
    )
    .orderBy("raceId", "avg_pit_stop_seconds")
)

display(q1)

### I first grouped the pit_stops data by raceId and driverId and used avg milliseconds to calculate the average pit stop time for each driver in each race. Then I grouped the data by raceId only and used min(milliseconds) and max(milliseconds) to get the fastest and slowest pit stop in each race. After that, I joined the two results using raceId so that each driver’s average pit stop time appears together with the overall fastest and slowest pit stop for that race. I also joined the result with the drivers dataset using driverId to include each driver’s first and last name. Finally, I divided the pit stop times by 1000 and used round(..., 3) to convert the values from milliseconds to seconds and make them easier to read.

#Q2: Rank order by finishing position the average time spent at the pit stop in each race.

In [0]:
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv",header=True,inferSchema=True)
display(df_results)

###For this question, I used the average pit stop time calculated in Q1 with the results dataset. The results table includes each driver’s finishing position in a race. So I used it to connect pit stop performance with race outcome. I joined the average pit stop data with the finishing results using both raceId and driverId. Then I ranked the results within each race using positionOrder. I kept the drivers as long as they still had a recorded positionOrder in the results dataset to deal with drivers who did not finish the race. This helps me keep the official race ordering instead of removing those cases by hand.

In [0]:
q2 = (
    q1
    .join(
        df_results.select("raceId", "driverId", "positionOrder"),
        on=["raceId", "driverId"],
        how="left"
    )
    .select(
        "raceId",
        "driverId",
        "forename",
        "surname",
        "positionOrder",
        "avg_pit_stop_seconds",
        "fastest_pit_stop_seconds",
        "slowest_pit_stop_seconds"
    )
    .orderBy("raceId", "positionOrder")
)

display(q2)

###I joined the Q1 result with the results dataset with both raceId and driverId. So each driver's pit stop summary could be matched with that driver's finishing position in the same race. I also used positionOrder from the results table to represent finishing position. Then I sorted the final output with orderBy("raceId", "positionOrder"). So the drivers appear in race-finishing order within each race. I kept the pit stop column so that the final table still shows the pit stop context together.

#Q3: Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

###For this question, I used the drivers dataset and checked the code column to see how missing values were stored. I checked for null values empty strings and \N as missing values in CSV files are not always shown as nulls. After identifying the missing codes, I filled them using a simple rule based on the driver's surname. I used the first three letters of the surname and removed any non-letter characters if needed. THen I converted the result to uppercase. If a driver already had a valid code, I kept the original value unchanged.

In [0]:
from pyspark.sql.functions import col

missing_code = (
    df_drivers
    .filter(
        col("code").isNull() |
        (col("code") == "") |
        (col("code") == r"\N")
    )
    .select("driverId", "forename", "surname", "code")
    .orderBy("driverId")
)

display(missing_code)

In [0]:
from pyspark.sql.functions import col, when, upper, substring, regexp_replace

q3 = (
    df_drivers
    .withColumn(
        "code_filled",
        when(
            col("code").isNull() | (col("code") == "") | (col("code") == r"\N"),
            upper(substring(regexp_replace(col("surname"), "[^A-Za-z]", ""), 1, 3))
        ).otherwise(col("code"))
    )
)

display(
    q3.select("driverId", "forename", "surname", "code", "code_filled")
    .orderBy("driverId")
)

###I first checked the code column in the drivers dataset and found that missing values could appear as nulls, empty strings, or \N. I included all three cases in my condition for detecting missing driver codes. Then I used withColumn() and when() to create a new column called code_filled. For missing values, I generated a replacement code from the driver's surname by removing non letter characters. If the original code was already present, I kept it unchanged with otherwise().

#Q4: Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".

In [0]:
df_races=spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv",header=True,inferSchema=True)
display(df_races)

###For this question, I used the results, drivers, and races datasets. I first linked each driver to the races. And then I combined that with each driver's date of birth and the date of the race. I defined age as the number of birthdays a driver had already had by the date. This is more accurate than just subtracting birth year from race year. I identified the youngest and oldest driver in each race by comparing the `Age` values within each race after calculating age for every driver race combination.

In [0]:
from pyspark.sql.functions import col, floor, months_between, to_date

age_table = (
    df_results
    .select("raceId", "driverId")
    .join(
        df_drivers.select("driverId", "forename", "surname", "dob"),
        on="driverId",
        how="left"
    )
    .join(
        df_races.select(
            "raceId",
            col("name").alias("race_name"),
            to_date(col("date")).alias("race_date")
        ),
        on="raceId",
        how="left"
    )
    .withColumn("dob", to_date(col("dob")))
    .withColumn("Age", floor(months_between(col("race_date"), col("dob")) / 12))
)

display(age_table)

In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

youngest_window = Window.partitionBy("raceId").orderBy(col("Age").asc(), col("driverId").asc())
oldest_window = Window.partitionBy("raceId").orderBy(col("Age").desc(), col("driverId").asc())

youngest_per_race = (
    age_table
    .withColumn("rn", row_number().over(youngest_window))
    .filter(col("rn") == 1)
    .select(
        "raceId",
        "race_name",
        col("driverId").alias("youngest_driverId"),
        col("forename").alias("youngest_forename"),
        col("surname").alias("youngest_surname"),
        col("Age").alias("youngest_age")
    )
)

oldest_per_race = (
    age_table
    .withColumn("rn", row_number().over(oldest_window))
    .filter(col("rn") == 1)
    .select(
        "raceId",
        col("driverId").alias("oldest_driverId"),
        col("forename").alias("oldest_forename"),
        col("surname").alias("oldest_surname"),
        col("Age").alias("oldest_age")
    )
)

q4 = (
    youngest_per_race
    .join(oldest_per_race, on="raceId", how="inner")
    .orderBy("raceId")
)

display(q4)

###I first created a age_table by joining the results, drivers, and races datasets. It is one row for each driver in each race. I created the Age column by using floor(months_between(race_date, dob) / 12). I used this because it measures the number of full years between the driver's birth date and the race date. It can match the number of birthdays the driver had already had. Next, I used two window functions partitioned by raceId. One sorted the drivers by Age in ascending order to find the youngest driver in each race. The other sorted by Age in descending order to find the oldest driver. I used row_number() to keep only the top row. Also, I joined the two results together into one final table.

#Q5: At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver

###For this question, I will the results and races` datasets. I first identified whether a driver finished 1st, 2nd, or 3rd in each race. Then I used a window function to calculate cumulative totals for each driver over time. I ordered the races chronologically and made sure the cumulative counts only included races before the current one. So that the current race result would not be counted in its own total.

In [0]:
from pyspark.sql.functions import col, when, sum, to_date
from pyspark.sql.window import Window

q5_base = (
    df_results
    .select("raceId", "driverId", "positionOrder")
    .join(
        df_races.select(
            "raceId",
            "year",
            "round",
            "name",
            to_date(col("date")).alias("race_date")
        ),
        on="raceId",
        how="left"
    )
    .join(
        df_drivers.select("driverId", "forename", "surname"),
        on="driverId",
        how="left"
    )
    .withColumn("win_flag", when(col("positionOrder") == 1, 1).otherwise(0))
    .withColumn("second_flag", when(col("positionOrder") == 2, 1).otherwise(0))
    .withColumn("third_flag", when(col("positionOrder") == 3, 1).otherwise(0))
)

podium_window = (
    Window
    .partitionBy("driverId")
    .orderBy("race_date", "raceId")
    .rowsBetween(Window.unboundedPreceding, -1)
)

q5 = (
    q5_base
    .withColumn("wins_before_race", sum("win_flag").over(podium_window))
    .withColumn("second_places_before_race", sum("second_flag").over(podium_window))
    .withColumn("third_places_before_race", sum("third_flag").over(podium_window))
    .fillna(0, subset=[
        "wins_before_race",
        "second_places_before_race",
        "third_places_before_race"
    ])
    .select(
        "raceId",
        "race_date",
        "name",
        "driverId",
        "forename",
        "surname",
        "positionOrder",
        "wins_before_race",
        "second_places_before_race",
        "third_places_before_race"
    )
    .orderBy("race_date", "driverId")
)

display(q5)

###I first joined the results, races, and drivers datasets. So, each row would represent one driver in one race together. Then I created three flag columns with when() to indicate whether the driver finished 1st, 2nd, or 3rd. Then I used a window function partitioned by driverId and ordered by race_date to process each driver’s race history in time order. I applied cumulative sum() to the three flag columns. I defined the window so that it ended at the row before the current one to make the totals represent previous races only. That is why drivers have zeros before their first race.

#Q6：Which Drivers Had the Most Wins in Each Season?

###For my own question, I wanted to find which drivers had the most race wins in each Formula 1 season. I used the results, races, and drivers datasets for this analysis. I first identified wins by selecting races where a driver finished in 1st place. Then I grouped the data by season and driver to count how many wins each driver had in that season. Finally, I ranked drivers within each season to find the driver or drivers with the highest number of wins.

In [0]:
from pyspark.sql.functions import col, count, dense_rank
from pyspark.sql.window import Window

q6_base = (
    df_results
    .filter(col("positionOrder") == 1)
    .join(
        df_races.select("raceId", "year"),
        on="raceId",
        how="left"
    )
    .join(
        df_drivers.select("driverId", "forename", "surname"),
        on="driverId",
        how="left"
    )
)

season_wins = (
    q6_base
    .groupBy("year", "driverId", "forename", "surname")
    .agg(count("*").alias("wins_in_season"))
)

season_rank_window = Window.partitionBy("year").orderBy(col("wins_in_season").desc())

q6 = (
    season_wins
    .withColumn("season_rank", dense_rank().over(season_rank_window))
    .filter(col("season_rank") == 1)
    .orderBy("year")
)

display(q6)

###I first filtered the results dataset to keep only races where a driver finished in 1st place. Then I joined the races dataset to get the season year and the drivers dataset to include driver names. I grouped the data by year and driver to count how many wins each driver had in each season. Finally, I used dense_rank() within each season to rank drivers by their total number of wins. I kept only the top-ranked driver or drivers for each year.